In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Iterator, List, Optional, Sequence, Tuple, Union

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import xgboost as xgb
import lightgbm as lgb
from lightgbm import LGBMClassifier
import shap

from src.data_prepatation import create_train_val_test_splits
from src.apnea_feature_tiers import get_features, filter_df, TIER_1, TIER_2, TIER_3

from src.data_prepatation import create_train_val_test_splits

### Configuration

In [ ]:
from config import dir_config
compiled_dir = dir_config.data.compiled
processed_dir = dir_config.data.processed

In [ ]:
random_seed = 2542
with_pca = False
target_future_step = 2
# target_future_step = 5

features = 'all' # options: 'all', 'tier1', 'tier12', 'tier123'
# features = 'tier1'
# features = 'tier12'
# features = 'tier123'

### Load Data

In [ ]:

# get all parquet files in the processed directory
parquet_files = list(Path(processed_dir).glob('*_with_metadata.parquet'))

# remove "agg_data_with_metadata.parquet" from the list of parquet files
parquet_files = [f for f in parquet_files if f.name != "agg_data_with_metadata.parquet"]

metadata = pd.read_csv(Path(processed_dir) / "participant_info.csv")

# find sid which have ahi < 5
non_apnea_sids = metadata[metadata['ahi'] < 5]['sid'].tolist()

# remove parquet files that correspond to non-apnea sids in their name (eg "{sid}_with_metadata.parquet")
parquet_files = [f for f in parquet_files if not any(sid in f.name for sid in non_apnea_sids)]

### Data processing

In [ ]:
train_X, train_y, fold_indices, val_X, val_y, test_X, test_y = create_train_val_test_splits(
    parquet_files,
    top_features=None,
    top_features_lag=0,
    target_type='apnea',
    target_future_steps=target_future_step,
    val_ratio = 0.15,
    test_ratio=0.2,
    n_splits=10,
    random_seed=random_seed,
)

Keep only tier 1 and tier 2 feature

In [ ]:
# # remove items containing "_range" or "_slope"
# TIER_1 = [feature for feature in TIER_1 if "_range" not in feature]
# TIER_2 = [feature for feature in TIER_2 if "_range" not in feature]
# TIER_3 = [feature for feature in TIER_3 if "_range" not in feature]

if features == 'all':
    columns_to_keep = train_X.columns.tolist()
elif features == 'tier1':
    columns_to_keep = TIER_1
elif features == 'tier12':
    columns_to_keep = TIER_1 + TIER_2
elif features == 'tier123':
    columns_to_keep = TIER_1 + TIER_2 + TIER_3
else:
    raise ValueError(f"Invalid value for features: {features}")

train_X = train_X[[col for col in train_X.columns if col in columns_to_keep]]
val_X = val_X[[col for col in val_X.columns if col in columns_to_keep]]
test_X = test_X[[col for col in test_X.columns if col in columns_to_keep]]

Perform PCA and transform X into pc space

In [ ]:
# Perform PCA and transform X into pc space with 95% variance explained in train_X
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

if with_pca:
    scaler = StandardScaler()
    train_X_scaled = scaler.fit_transform(train_X)
    pca = PCA(random_state=random_seed)
    pca.fit(train_X_scaled)
    n_components_95 = np.argmax(pca.explained_variance_ratio_.cumsum() >= 0.95) + 1

    scaler = StandardScaler()
    train_X_scaled = scaler.fit_transform(train_X)
    val_X_scaled = scaler.transform(val_X)
    test_X_scaled = scaler.transform(test_X)

    train_X_pca = pca.transform(train_X_scaled)[:, :n_components_95]
    val_X_pca = pca.transform(val_X_scaled)[:, :n_components_95]
    test_X_pca = pca.transform(test_X_scaled)[:, :n_components_95]

In [ ]:
if with_pca:
    plt.plot(pca.explained_variance_ratio_.cumsum())
    plt.xlabel("Number of Components")
    plt.ylabel("Cumulative Explained Variance")
    plt.title("PCA Explained Variance")
    plt.show()


In [ ]:
# save data as a pickle file
import pickle

if with_pca:
    filename = f"apnea_detection_features_{features}_pc_{5*target_future_step}secs_future.pkl"

    with open(Path(processed_dir) / f"{filename}", "wb") as f:
        pickle.dump({
            "train_X": train_X_pca,
            "train_y": train_y,
            "fold_indices": fold_indices,
            "val_X": val_X_pca,
            "val_y": val_y,
            "test_X": test_X_pca,
            "test_y": test_y,
            "train_pca": pca,
            "train_scaler": scaler,
        }, f)

else:
    filename = f"apnea_detection_features_{features}_{5*target_future_step}secs_future.pkl"

    with open(Path(processed_dir) / f"{filename}", "wb") as f:
        pickle.dump({
            "train_X": train_X,
            "train_y": train_y,
            "fold_indices": fold_indices,
            "val_X": val_X,
            "val_y": val_y,
            "test_X": test_X,
            "test_y": test_y,
        }, f)

### Test Stratification of data